## MemPalace — Databricks-Native Setup

One-time infrastructure provisioning. Creates:

1. **Schema** — `scratch.llm`
2. **Delta tables** — drawers, entities, triples, diaries, WAL
3. **Volume** — `mempalace_config` for JSON/text config files
4. **Vector Search endpoint** — `mempalace_vs_endpoint`
5. **Vector Search index** — `scratch.llm.mempalace_drawers_index` (Delta Sync, managed embeddings)

Idempotent — safe to re-run.

In [0]:
%pip install databricks-vectorsearch --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
import time

# ── Target catalog / schema ───────────────────────────────────────────────────
CATALOG = "scratch"
SCHEMA = "llm"
FQ_PREFIX = f"{CATALOG}.{SCHEMA}"  # scratch.llm

# ── Table names ───────────────────────────────────────────────────────────────
DRAWERS_TABLE     = f"{FQ_PREFIX}.mempalace_drawers"
ENTITIES_TABLE    = f"{FQ_PREFIX}.mempalace_entities"
TRIPLES_TABLE     = f"{FQ_PREFIX}.mempalace_triples"
DIARIES_TABLE     = f"{FQ_PREFIX}.mempalace_diaries"
WAL_TABLE         = f"{FQ_PREFIX}.mempalace_wal"

# ── Vector Search ─────────────────────────────────────────────────────────────
VS_ENDPOINT       = "mempalace_vs_endpoint"
VS_INDEX          = f"{FQ_PREFIX}.mempalace_drawers_index"
EMBEDDING_MODEL   = "databricks-gte-large-en"  # Databricks-managed, production-grade

# ── Volume ────────────────────────────────────────────────────────────────────
CONFIG_VOLUME     = f"{FQ_PREFIX}.mempalace_config"
CONFIG_VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/mempalace_config"

print(f"[setup] Target:     {FQ_PREFIX}")
print(f"[setup] Drawers:    {DRAWERS_TABLE}")
print(f"[setup] VS Index:   {VS_INDEX}")
print(f"[setup] VS Endpoint: {VS_ENDPOINT}")
print(f"[setup] Config Vol: {CONFIG_VOLUME_PATH}")

### Stage 1 — Schema, Tables & Volume

Creates the `scratch.llm` schema (if needed), all five Delta tables, and the config Volume.
All statements are idempotent (`IF NOT EXISTS`).

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS scratch.llm
COMMENT 'MemPalace Databricks-native memory system';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS scratch.llm.mempalace_drawers (
    id              STRING        NOT NULL,
    text            STRING        NOT NULL,
    wing            STRING        NOT NULL,
    room            STRING        NOT NULL,
    hall            STRING,
    source_file     STRING,
    source_mtime    DOUBLE,
    chunk_index     INT,
    date            STRING,
    importance      DOUBLE        DEFAULT 3.0,
    agent           STRING        DEFAULT 'mempalace',
    filed_at        TIMESTAMP     DEFAULT current_timestamp(),
    CONSTRAINT pk_drawers PRIMARY KEY (id)
)
USING DELTA
COMMENT 'Verbatim text drawers — the core memory store'
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true'
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS scratch.llm.mempalace_entities (
    id              STRING        NOT NULL,
    name            STRING        NOT NULL,
    type            STRING        DEFAULT 'unknown',
    properties      STRING        DEFAULT '{}',
    created_at      TIMESTAMP     DEFAULT current_timestamp(),
    CONSTRAINT pk_entities PRIMARY KEY (id)
)
USING DELTA
COMMENT 'Knowledge graph entity nodes (people, projects, tools, concepts)';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS scratch.llm.mempalace_triples (
    id              STRING        NOT NULL,
    subject         STRING        NOT NULL,
    predicate       STRING        NOT NULL,
    object          STRING        NOT NULL,
    valid_from      STRING,
    valid_to        STRING,
    confidence      DOUBLE        DEFAULT 1.0,
    source_closet   STRING,
    source_file     STRING,
    extracted_at    TIMESTAMP     DEFAULT current_timestamp(),
    CONSTRAINT pk_triples PRIMARY KEY (id)
)
USING DELTA
COMMENT 'Knowledge graph temporal relationship triples';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS scratch.llm.mempalace_diaries (
    id              STRING        NOT NULL,
    agent_name      STRING        NOT NULL,
    entry           STRING        NOT NULL,
    written_at      TIMESTAMP     DEFAULT current_timestamp(),
    CONSTRAINT pk_diaries PRIMARY KEY (id)
)
USING DELTA
COMMENT 'AAAK-encoded agent diary entries';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS scratch.llm.mempalace_wal (
    timestamp       TIMESTAMP     NOT NULL,
    operation       STRING        NOT NULL,
    params          STRING        NOT NULL,
    result          STRING,
    caller          STRING        DEFAULT current_user()
)
USING DELTA
COMMENT 'Write-ahead audit log — append-only';

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS scratch.llm.mempalace_config
COMMENT 'JSON/text config files for MemPalace (identity, wing config, people map)';

### Stage 2 — Vector Search Endpoint & Index

Creates the VS endpoint (if it doesn’t already exist) and a **Delta Sync index**
with Databricks-managed embeddings (`databricks-gte-large-en`).

The index uses **TRIGGERED** sync — call `index.sync()` after batch ingests.

In [0]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

# ── Create endpoint (skip if it already exists) ─────────────────────────
try:
    ep = vsc.get_endpoint(VS_ENDPOINT)
    print(f"[setup] Endpoint '{VS_ENDPOINT}' already exists (status: {ep.get('endpoint_status', {}).get('state', 'UNKNOWN')})")
except Exception:
    print(f"[setup] Creating endpoint '{VS_ENDPOINT}'...")
    vsc.create_endpoint(
        name=VS_ENDPOINT,
        endpoint_type="STANDARD",
    )
    print(f"[setup] Endpoint '{VS_ENDPOINT}' creation initiated.")

# ── Wait for endpoint to be ONLINE ───────────────────────────────────
print(f"[setup] Waiting for endpoint to be ONLINE...")
t0 = time.perf_counter()
while True:
    ep = vsc.get_endpoint(VS_ENDPOINT)
    state = ep.get("endpoint_status", {}).get("state", "UNKNOWN")
    if state == "ONLINE":
        elapsed = time.perf_counter() - t0
        print(f"[setup] Endpoint ONLINE [{elapsed:.1f}s]")
        break
    if state in ("FAILED", "DELETED"):
        raise RuntimeError(f"Endpoint entered {state} state: {ep}")
    time.sleep(15)

In [0]:
# ── Create the Delta Sync index (skip if it already exists) ───────────
try:
    existing_index = vsc.get_index(
        endpoint_name=VS_ENDPOINT,
        index_name=VS_INDEX,
    )
    print(f"[setup] Index '{VS_INDEX}' already exists.")
    print(f"        Status: {existing_index.describe().get('status', {})}")
except Exception:
    print(f"[setup] Creating Delta Sync index '{VS_INDEX}'...")
    print(f"        Source table:    {DRAWERS_TABLE}")
    print(f"        Embedding model: {EMBEDDING_MODEL}")
    print(f"        Sync mode:       TRIGGERED")

    vsc.create_delta_sync_index(
        endpoint_name=VS_ENDPOINT,
        source_table_name=DRAWERS_TABLE,
        index_name=VS_INDEX,
        pipeline_type="TRIGGERED",
        primary_key="id",
        embedding_source_column="text",
        embedding_model_endpoint_name=EMBEDDING_MODEL,
        columns_to_sync=[
            "id", "text", "wing", "room", "hall",
            "source_file", "date", "importance", "filed_at",
        ],
    )
    print(f"[setup] Index creation initiated.")

# ── Wait for index to be ONLINE ─────────────────────────────────────
print(f"[setup] Waiting for index to be ready...")
t0 = time.perf_counter()
while True:
    idx = vsc.get_index(
        endpoint_name=VS_ENDPOINT,
        index_name=VS_INDEX,
    )
    status = idx.describe().get("status", {}).get("ready", False)
    if status:
        elapsed = time.perf_counter() - t0
        print(f"[setup] Index READY [{elapsed:.1f}s]")
        break
    detail = idx.describe().get("status", {})
    msg = detail.get("message", "")
    if "FAILED" in str(detail).upper():
        raise RuntimeError(f"Index creation failed: {detail}")
    time.sleep(30)

### Stage 3 — Default Config Files

Writes default `config.json`, `identity.txt`, and `wing_config.json` to the
config Volume. Skips any file that already exists.

In [0]:
import json
from pathlib import Path

vol = Path(CONFIG_VOLUME_PATH)
vol.mkdir(parents=True, exist_ok=True)

# ── config.json ──────────────────────────────────────────────────────────────
config_path = vol / "config.json"
if not config_path.exists():
    default_config = {
        "catalog": CATALOG,
        "schema": SCHEMA,
        "vs_endpoint": VS_ENDPOINT,
        "embedding_model": EMBEDDING_MODEL,
        "topic_wings": [
            "emotions", "consciousness", "memory",
            "technical", "identity", "family", "creative",
        ],
    }
    config_path.write_text(json.dumps(default_config, indent=2), encoding="utf-8")
    print(f"[setup] Wrote {config_path}")
else:
    print(f"[setup] {config_path} already exists — skipped")

# ── identity.txt (Layer 0) ───────────────────────────────────────────────
identity_path = vol / "identity.txt"
if not identity_path.exists():
    identity_path.write_text(
        "I am a MemPalace AI assistant.\n"
        "Edit this file to set your Layer 0 identity.",
        encoding="utf-8",
    )
    print(f"[setup] Wrote {identity_path}")
else:
    print(f"[setup] {identity_path} already exists — skipped")

# ── wing_config.json ─────────────────────────────────────────────────────
wing_config_path = vol / "wing_config.json"
if not wing_config_path.exists():
    wing_config = {
        "default_wing": "wing_general",
        "wings": {},
    }
    wing_config_path.write_text(json.dumps(wing_config, indent=2), encoding="utf-8")
    print(f"[setup] Wrote {wing_config_path}")
else:
    print(f"[setup] {wing_config_path} already exists — skipped")

# ── people_map.json ──────────────────────────────────────────────────────
people_map_path = vol / "people_map.json"
if not people_map_path.exists():
    people_map_path.write_text(json.dumps({}, indent=2), encoding="utf-8")
    print(f"[setup] Wrote {people_map_path}")
else:
    print(f"[setup] {people_map_path} already exists — skipped")

print("\n[setup] Config Volume ready.")

### Stage 4 — Validation

Inserts a test drawer, triggers a Vector Search sync, queries for it,
then cleans up. Confirms the full write → embed → search pipeline works.

In [0]:
import hashlib
from datetime import datetime

# ── Insert a test drawer ───────────────────────────────────────────────────
TEST_ID = hashlib.sha256(b"__mempalace_setup_test__").hexdigest()
TEST_TEXT = (
    "This is a MemPalace setup validation drawer. "
    "The ancient Greek method of loci places memories in rooms of an imaginary palace. "
    "If you can find this via Vector Search, the pipeline is working."
)

spark.sql(f"""
    MERGE INTO {DRAWERS_TABLE} AS target
    USING (SELECT
        '{TEST_ID}' AS id,
        '{TEST_TEXT}' AS text,
        'wing_setup_test' AS wing,
        'validation' AS room,
        'hall_facts' AS hall,
        '__setup_notebook__' AS source_file,
        NULL AS source_mtime,
        0 AS chunk_index,
        '{datetime.utcnow().strftime("%Y-%m-%d")}' AS date,
        5.0 AS importance,
        'setup' AS agent,
        current_timestamp() AS filed_at
    ) AS source
    ON target.id = source.id
    WHEN NOT MATCHED THEN INSERT *
""")
print(f"[validate] Test drawer inserted (id: {TEST_ID[:16]}...)")

# ── Trigger VS index sync ────────────────────────────────────────────────
print("[validate] Triggering VS index sync...")
index = vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=VS_INDEX)
index.sync()

# Wait for sync to complete
t0 = time.perf_counter()
while True:
    status = index.describe().get("status", {})
    if status.get("ready", False):
        # Check if there are indexed rows
        num_rows = status.get("num_rows_updated", 0) or status.get("indexed_row_count", 0)
        if num_rows and num_rows > 0:
            elapsed = time.perf_counter() - t0
            print(f"[validate] Sync complete — {num_rows} rows indexed [{elapsed:.1f}s]")
            break
    if "FAILED" in str(status).upper():
        raise RuntimeError(f"Index sync failed: {status}")
    time.sleep(15)

# ── Query the index for the test drawer ─────────────────────────────────
print("[validate] Querying VS index for 'method of loci palace memories'...")
results = index.similarity_search(
    query_text="method of loci palace memories",
    columns=["id", "text", "wing", "room"],
    num_results=3,
)

hits = results.get("result", {}).get("data_array", [])
found = any(row[0] == TEST_ID for row in hits)

if found:
    print("[validate] ✅ SUCCESS — test drawer found via Vector Search!")
else:
    print("[validate] ⚠️  Test drawer not in top-3 results (may need more sync time).")
    print(f"           Hits: {hits}")

# ── Clean up the test drawer ─────────────────────────────────────────────
spark.sql(f"DELETE FROM {DRAWERS_TABLE} WHERE id = '{TEST_ID}'")
print(f"[validate] Test drawer cleaned up.")

---
### Setup Complete

**Infrastructure created:**

| Resource | Name |
| --- | --- |
| Schema | `scratch.llm` |
| Drawers table | `scratch.llm.mempalace_drawers` |
| Entities table | `scratch.llm.mempalace_entities` |
| Triples table | `scratch.llm.mempalace_triples` |
| Diaries table | `scratch.llm.mempalace_diaries` |
| WAL table | `scratch.llm.mempalace_wal` |
| Config Volume | `/Volumes/scratch/llm/mempalace_config/` |
| VS Endpoint | `mempalace_vs_endpoint` |
| VS Index | `scratch.llm.mempalace_drawers_index` |

**Next steps:** Proceed with Phase 2 — rewrite `searcher.py` and `layers.py` to use Vector Search.